# 🌦️ Weather & Climate Sentiment Analysis
---
> An end-to-end **NLP & Machine Learning** project that analyzes public sentiment on climate change using NASA comment data.

| Step | Description |
|------|-------------|
| 📥 Data Loading | Read NASA climate CSV dataset |
| 🧹 Text Cleaning | Lowercase, remove noise, lemmatize |
| 🏷️ Sentiment Labeling | TextBlob polarity → Positive / Negative / Neutral |
| 🔢 Vectorization | TF-IDF (max 5000 features) |
| 🤖 ML Model | Multinomial Naive Bayes classifier |
| 📊 Evaluation | Accuracy score + visualization |


## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print('✅ All libraries imported!')

## 📂 2. Load Dataset

> **Note:** Update the CSV path below to match your file location.

In [ ]:
df = pd.read_csv('climate_nasa.csv')  # Update path if needed
df.columns = df.columns.str.strip()

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

## 🔍 3. Explore the Dataset

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)
print('\nSample text entries:')
df['text'].dropna().head(5)

## 🧹 4. Text Preprocessing

Steps applied to each comment:
- Convert to **lowercase**
- Remove **special characters and numbers**
- Remove **stopwords**
- Apply **lemmatization** (reduce words to root form)

In [ ]:
# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

# Drop rows with missing text
df = df.dropna(subset=['text'])
df['clean_text'] = df['text'].apply(clean_text)

print('✅ Text cleaning complete!')
print('\nSample cleaned text:')
df[['text','clean_text']].head(3)

## 🏷️ 5. Sentiment Labeling with TextBlob

Using **TextBlob polarity score**:
- `polarity > 0` → **Positive** 😊
- `polarity < 0` → **Negative** 😠
- `polarity == 0` → **Neutral** 😐

In [ ]:
def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return 'Positive'
    elif polarity < 0:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['text'].apply(get_sentiment)

print('Sentiment Distribution:')
print(df['Sentiment'].value_counts())
print(f'\nTotal labeled rows: {len(df)}')

### 5.1 Visualize Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#27AE60', '#E74C3C', '#3498DB']
df['Sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Sentiment Distribution (Bar)', fontsize=13)
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Pie chart
df['Sentiment'].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Sentiment Distribution (Pie)', fontsize=13)
axes[1].set_ylabel('')

plt.suptitle('Climate Change Comment Sentiments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔢 6. TF-IDF Vectorization

Converts cleaned text to numerical features using **Term Frequency-Inverse Document Frequency**.

In [ ]:
X = df['clean_text']
y = df['Sentiment']

vectorizer = TfidfVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(X)

print('Feature matrix shape:', X_vec.shape)
print('Number of classes:', y.nunique())
print('Class labels:', y.unique().tolist())

## ✂️ 7. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## 🤖 8. Train Multinomial Naive Bayes Model

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'\n✅ Model Accuracy: {round(accuracy * 100, 2)}%')

## 📋 9. Model Evaluation

In [ ]:
print('Classification Report:')
print(classification_report(y_test, y_pred))

# Accuracy bar
plt.figure(figsize=(5, 3))
plt.barh(['Naive Bayes'], [accuracy * 100], color='#2E86AB', height=0.4)
plt.xlim(0, 100)
plt.xlabel('Accuracy %')
plt.title('Model Accuracy')
for i, v in enumerate([accuracy * 100]):
    plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=12)
plt.tight_layout()
plt.show()

## 💾 10. Save Output CSV

In [ ]:
df.to_csv('climate_output_with_sentiment.csv', index=False)
print('✅ Output saved as: climate_output_with_sentiment.csv')
print(f'Total rows saved: {len(df)}')
df[['text', 'Sentiment', 'clean_text']].head()

## 🔮 11. Predict Sentiment on Custom Text

In [ ]:
def predict_sentiment(custom_text):
    cleaned = clean_text(custom_text)
    vec = vectorizer.transform([cleaned])
    prediction = model.predict(vec)[0]
    polarity = round(TextBlob(custom_text).sentiment.polarity, 3)
    print(f'Text     : {custom_text}')
    print(f'Polarity : {polarity}')
    print(f'Sentiment: {prediction}')
    return prediction

# Test examples
print('--- Example Predictions ---\n')
predict_sentiment('Climate change is a serious threat to our planet and we must act now!')
print()
predict_sentiment('The weather data shows normal seasonal variation nothing alarming here')
print()
predict_sentiment('Renewable energy is growing and giving hope for a cleaner future')

## ✅ 12. Conclusion

| Metric | Value |
|--------|-------|
| Dataset | NASA Climate Change Comments |
| Total Records | ~500+ |
| Vectorizer | TF-IDF (5000 features) |
| Model | Multinomial Naive Bayes |
| Train/Test Split | 80% / 20% |
| Sentiment Classes | Positive, Negative, Neutral |

**Key Takeaways:**
- Neutral comments dominate the dataset — many factual/scientific statements
- Negative sentiments reflect concern and frustration around climate inaction
- Positive sentiments highlight hope in renewable energy and scientific progress
- Naive Bayes with TF-IDF is a fast and effective baseline for text classification

> 🌦️ *Project completed successfully!*
